# districtJudgeTest v8 (260920) — 군구 필터 복원 검증

| 항목 | 내용 |
| --- | --- |
| 작성 일자 | 2026-09-20 |
| 선행 노트북 | `avoidZoneTest_v6_260827.ipynb`, `avoidZoneTest_v7_260920.ipynb` |
| Tmap 호출 | **0회** (좌표 연산만 수행) |
| 산출물 | `./output8_260920/` |

## 1. 배경

서버 `algorithm.py`에는 군구 필터가 살아 있으나, 앱이 `district`를 보내지 않아
`subset`이 항상 비고 폴백이 발동한다. 그 결과 인천 전체 핫스팟 1,535곳이 대상이 되며,
강화군 Buffer 반경 2.5km 특례도 작동하지 않는다(코드 조사 F-12).

v6 실측에서 이 조건은 v5(필터 적용) 대비 hard case가 9.4%p 높았다(K=2 기준 75.0% vs 65.6%).
공업지대 통과 거리 총 감소율은 24.4% vs 25.3%로 거의 같으므로, 차이는 회피의 크기가 아니라
완결성에 있다.

앱 재배포 없이 필터를 복원하기 위해, **서버가 `origin` 좌표로 군구를 판정하는 방식**을 택한다.
본 노트북은 그 방식이 안전한지 검증한다.

## 2. 검증 대상

세 가지 판정 방식을 비교한다.

| 방식 | 내용 |
| --- | --- |
| **M1** | 출발지에서 가장 가까운 핫스팟 1곳의 `source_district` |
| **M2** | 가장 가까운 핫스팟 10곳의 다수결 |
| **M3** | 반경 1km 내 핫스팟의 다수결. 없으면 M1로 대체 |

경계 인근에서 M1은 단일 표본에 의존하므로 흔들릴 수 있다. M2·M3는 그 완화안이다.

## 3. 판정 기준

**주 지표.** 좌표 판정으로 선정한 핫스팟 2곳이, 라벨을 썼을 때와 동일한가.

| 선정 일치율 | 해석 |
| --- | --- |
| 95% 이상 | 좌표 판정 채택 |
| 85~95% | 판정 방식 조정 후 재검토 |
| 85% 미만 | 앱이 `district`를 보내는 방식(방법 A)으로 전환 |

**부 지표.** 군구 라벨 일치율, Buffer 후보 수, K=2 충족률, 폴백 발동률.

라벨 일치율은 참고용이다. v6 표본의 `district`는 표본 생성 조건이지 행정 경계로 확정한
정답이 아니며, 특히 분리군은 군구별 bounding box에서 출발지를 뽑았으므로 이웃 구에
걸칠 수 있다. 불일치가 곧 판정 실패를 뜻하지 않는다.

## 4. 함께 점검하는 사항

- 강화군 Buffer 반경 2.5km 특례가 복원되는가
- 필터 적용 시 후보 부족(K=2 미충족)이 발생하는가
- 인천 경계 밖 좌표가 들어왔을 때를 가려낼 임계값 (미결 E그룹 F-07~F-11)

## 5. 실행 전 준비

```
./hotspots_CCTV_20260510.csv
./output/incheon_avoid_zones_v1_260601.geojson
./output6_260827/mixed_od_200.pkl
```

`.env`는 필요 없다.

In [1]:
# 셀 1: 라이브러리 import

import os
import ast
import json
import math
import time
import pickle
import itertools
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from shapely.geometry import shape, Point, LineString
from shapely.ops import unary_union
from shapely.prepared import prep
from scipy.spatial import cKDTree

print("[v8 노트북 시작 — 군구 필터 복원 검증]")
print(f"  pandas {pd.__version__} / numpy {np.__version__}")
print(f"  작업 디렉토리: {os.getcwd()}")
print("  Tmap 호출: 없음")

[v8 노트북 시작 — 군구 필터 복원 검증]
  pandas 3.0.3 / numpy 2.4.6
  작업 디렉토리: c:\Users\User\Documents\GitHub\ploggo-algorithm-test
  Tmap 호출: 없음


In [2]:
# 셀 2: 경로 상수 + 입력 파일 점검

CSV_PATH     = Path("./hotspots_CCTV_20260510.csv")
GEOJSON_PATH = Path("./output/incheon_avoid_zones_v1_260601.geojson")
OD_PKL_PATH  = Path("./output6_260827/mixed_od_200.pkl")

STAMP   = "260920"
V8_ROOT = Path(f"./output8_{STAMP}")
V8_ROOT.mkdir(parents=True, exist_ok=True)

_missing = [p for p in (CSV_PATH, GEOJSON_PATH, OD_PKL_PATH) if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "입력 파일이 없다:\n  " + "\n  ".join(str(p.resolve()) for p in _missing))

print("[입력 파일 확인]")
for p in (CSV_PATH, GEOJSON_PATH, OD_PKL_PATH):
    print(f"  {p}  ({p.stat().st_size/1024:.0f} KB)")
print(f"[출력 폴더] {V8_ROOT.resolve()}")

[입력 파일 확인]
  hotspots_CCTV_20260510.csv  (334 KB)
  output\incheon_avoid_zones_v1_260601.geojson  (1719 KB)
  output6_260827\mixed_od_200.pkl  (1347 KB)
[출력 폴더] C:\Users\User\Documents\GitHub\ploggo-algorithm-test\output8_260920


In [3]:
# 셀 3: 회피 영역 로드 (v6 셀 3 이식)

with open(GEOJSON_PATH, "r", encoding="utf-8") as f:
    avoid_geojson = json.load(f)

avoid_polygons = [
    {"geom": shape(ft["geometry"]),
     "uqa_code": ft["properties"].get("uqa_code"),
     "sigg_name": ft["properties"].get("sigg_name")}
    for ft in avoid_geojson["features"]
]
avoid_union = unary_union([p["geom"] for p in avoid_polygons])
avoid_union_prepared = prep(avoid_union)

print(f"[회피 영역 로드] 폴리곤 {len(avoid_polygons)}개")

[회피 영역 로드] 폴리곤 147개


In [4]:
# 셀 4: 핫스팟 로드 + S_i + in_avoid_zone (v6 셀 4 이식)

EARTH_RADIUS_KM = 6371.0

hotspots = pd.read_csv(CSV_PATH, encoding="utf-8-sig")


def parse_list_field(v):
    if pd.isna(v):
        return None
    if isinstance(v, str):
        try:
            return ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return None
    return v


for col in ("install_years", "relocation_history"):
    if col in hotspots.columns:
        hotspots[col] = hotspots[col].apply(parse_list_field)

_lat_rad = np.radians(hotspots["latitude"].values)
_lon_rad = np.radians(hotspots["longitude"].values)
_xyz = np.column_stack([
    np.cos(_lat_rad) * np.cos(_lon_rad),
    np.cos(_lat_rad) * np.sin(_lon_rad),
    np.sin(_lat_rad),
])
_tree = cKDTree(_xyz)
hotspots["neighbor_200m"] = _tree.query_ball_point(
    _xyz, r=2 * np.sin((0.2 / EARTH_RADIUS_KM) / 2), return_length=True) - 1
hotspots["S_i"] = hotspots["cctv_count"] + 0.5 * hotspots["neighbor_200m"]
hotspots["in_avoid_zone"] = hotspots.apply(
    lambda r: avoid_union_prepared.contains(Point(r["longitude"], r["latitude"])), axis=1)

print(f"[핫스팟 로드] {len(hotspots):,}곳")
print(f"  군구별 분포:")
for d, n in hotspots["source_district"].value_counts().items():
    print(f"    {d:<12} {n:>5}곳")

[핫스팟 로드] 1,535곳
  군구별 분포:
    seogu          379곳
    gyeyang        292곳
    michuhol       230곳
    yeonsu         229곳
    namdong        155곳
    ganghwa         86곳
    junggu          74곳
    donggu          48곳
    bupyeong        42곳


In [5]:
# 셀 5: 거리·선정 유틸 (v6 이식, 군구별 Buffer 반경 복원)

DETOUR_RATIO = 1.3
ALPHA, BETA, GAMMA = 0.5, 0.5, 1.0
K, BF_TOP_N = 2, 15


def haversine_km(p_a, p_b):
    lat1, lon1 = p_a
    lat2, lon2 = p_b
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return EARTH_RADIUS_KM * 2 * math.asin(math.sqrt(a))


def haversine_vec(p, coords):
    """p(lat,lon) 와 coords(N,2) 사이의 거리 배열 (km)"""
    lat1, lon1 = math.radians(p[0]), math.radians(p[1])
    lat2 = np.radians(coords[:, 0])
    lon2 = np.radians(coords[:, 1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + math.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return EARTH_RADIUS_KM * 2 * np.arcsin(np.sqrt(a))


def walking_distance_km(p_a, p_b):
    return haversine_km(p_a, p_b) * DETOUR_RATIO


def detour_distance_km(origin, hotspot, destination):
    return max(walking_distance_km(origin, hotspot)
               + walking_distance_km(hotspot, destination)
               - walking_distance_km(origin, destination), 0.0)


def get_buffer_radius(district):
    """군구별 Buffer 반경 (km). 강화군만 2.5km — v5 사양 복원"""
    return 2.5 if district == "ganghwa" else 1.5


def ellipse_filter(df, origin, destination, buffer_r_km):
    d_od = walking_distance_km(origin, destination)
    threshold = buffer_r_km * 1.5
    coords = df[["latitude", "longitude"]].values
    d_oh = haversine_vec(origin, coords) * DETOUR_RATIO
    d_hd = haversine_vec(destination, coords) * DETOUR_RATIO
    return df[(d_oh + d_hd - d_od) < threshold].copy()


def compute_avoid_penalty(h_lat, h_lon, origin, destination):
    if avoid_union_prepared.contains(Point(h_lon, h_lat)):
        return 999.0
    lo = LineString([(origin[1], origin[0]), (h_lon, h_lat)]).intersection(avoid_union)
    ld = LineString([(h_lon, h_lat), (destination[1], destination[0])]).intersection(avoid_union)
    return ((lo.length if not lo.is_empty else 0)
            + (ld.length if not ld.is_empty else 0)) * 111.0


def select_hotspots(cands, origin, destination, k=K):
    """S_i 상위 BF_TOP_N 풀 → Score_i 상위 K 선정 (v6과 동일)"""
    n = len(cands)
    if n == 0:
        return None
    pool = cands.nlargest(min(BF_TOP_N, n), "S_i").copy()
    pool["D_i"] = [detour_distance_km(origin, (r.latitude, r.longitude), destination)
                   for r in pool.itertuples()]
    pool["avoid_penalty_i"] = [compute_avoid_penalty(r.latitude, r.longitude, origin, destination)
                               for r in pool.itertuples()]
    pool["Score_i"] = ALPHA * pool["S_i"] - BETA * pool["D_i"] - GAMMA * pool["avoid_penalty_i"]
    return pool.nlargest(min(k, n), "Score_i")


print("[유틸 정의 완료]")
print(f"  α={ALPHA} β={BETA} γ={GAMMA} K={K} BF_TOP_N={BF_TOP_N}")
print(f"  Buffer 반경: 일반 1.5km / 강화군 2.5km (v5 사양 복원)")

[유틸 정의 완료]
  α=0.5 β=0.5 γ=1.0 K=2 BF_TOP_N=15
  Buffer 반경: 일반 1.5km / 강화군 2.5km (v5 사양 복원)


In [6]:
# 셀 6: 군구 판정 함수 3종
# 서버가 origin 좌표만으로 군구를 정하는 방식의 후보들이다.

HS_COORDS    = hotspots[["latitude", "longitude"]].values
HS_DISTRICTS = hotspots["source_district"].values

KNN_N        = 10     # M2: 최근접 몇 곳의 다수결인가
RADIUS_KM    = 1.0    # M3: 반경 몇 km 내 다수결인가


def judge_m1(origin):
    """최근접 핫스팟 1곳의 군구"""
    d = haversine_vec(origin, HS_COORDS)
    i = int(d.argmin())
    return HS_DISTRICTS[i], float(d[i])


def judge_m2(origin, n=KNN_N):
    """최근접 n곳의 다수결. 동률이면 더 가까운 쪽을 따른다."""
    d = haversine_vec(origin, HS_COORDS)
    idx = np.argpartition(d, min(n, len(d) - 1))[:n]
    idx = idx[np.argsort(d[idx])]
    votes = Counter(HS_DISTRICTS[idx])
    top = max(votes.values())
    tied = {k for k, v in votes.items() if v == top}
    for i in idx:                       # 가까운 순으로 훑어 동률 해소
        if HS_DISTRICTS[i] in tied:
            return HS_DISTRICTS[i], float(d[idx[0]])
    return HS_DISTRICTS[idx[0]], float(d[idx[0]])


def judge_m3(origin, radius_km=RADIUS_KM):
    """반경 내 다수결. 반경 안에 하나도 없으면 M1로 대체한다."""
    d = haversine_vec(origin, HS_COORDS)
    mask = d <= radius_km
    if not mask.any():
        return judge_m1(origin)
    sub_d = d[mask]
    sub_dist = HS_DISTRICTS[mask]
    votes = Counter(sub_dist)
    top = max(votes.values())
    tied = {k for k, v in votes.items() if v == top}
    order = np.argsort(sub_d)
    for i in order:
        if sub_dist[i] in tied:
            return sub_dist[i], float(d.min())
    return sub_dist[order[0]], float(d.min())


# 동작 확인 — 각 군구 핫스팟의 중심 좌표를 넣어 자기 군구가 나오는지 본다
print("[판정 함수 동작 확인 — 군구별 핫스팟 중심 좌표 입력]")
print(f"  {'군구':<12} {'M1':<12} {'M2':<12} {'M3':<12} {'최근접(m)':>10}")
print("-" * 62)
for d_name, sub in hotspots.groupby("source_district"):
    c = (sub["latitude"].mean(), sub["longitude"].mean())
    m1, dist_km = judge_m1(c)
    m2, _ = judge_m2(c)
    m3, _ = judge_m3(c)
    flag = "" if (m1 == m2 == m3 == d_name) else "  <- 불일치"
    print(f"  {d_name:<12} {m1:<12} {m2:<12} {m3:<12} {dist_km*1000:>9.0f}{flag}")

[판정 함수 동작 확인 — 군구별 핫스팟 중심 좌표 입력]
  군구           M1           M2           M3               최근접(m)
--------------------------------------------------------------
  bupyeong     bupyeong     bupyeong     bupyeong           488
  donggu       donggu       donggu       donggu              80
  ganghwa      ganghwa      ganghwa      ganghwa           1134
  gyeyang      gyeyang      gyeyang      gyeyang            233
  junggu       junggu       junggu       junggu            1608
  michuhol     michuhol     michuhol     michuhol            65
  namdong      namdong      namdong      namdong            345
  seogu        seogu        seogu        seogu              338
  yeonsu       yeonsu       yeonsu       yeonsu             192


In [7]:
# 셀 7: OD 200쌍 로드 + 판정 실행 + 라벨 일치율

with open(OD_PKL_PATH, "rb") as f:
    od_list = pickle.load(f)

print(f"[OD 로드] {len(od_list)}쌍 "
      f"(인접 {sum(1 for o in od_list if o.get('group')=='adjacent')} / "
      f"분리 {sum(1 for o in od_list if o.get('group')=='separated')})")

t0 = time.time()
for od in od_list:
    od["m1"], od["nearest_km"] = judge_m1(od["O"])
    od["m2"], _ = judge_m2(od["O"])
    od["m3"], _ = judge_m3(od["O"])
print(f"[판정 완료] 소요 {time.time()-t0:.1f}초")

lab = np.array([o.get("district") for o in od_list])
print("\n[라벨 일치율 — 참고 지표]")
for m in ("m1", "m2", "m3"):
    got = np.array([o[m] for o in od_list])
    acc = (got == lab).mean() * 100
    print(f"  {m.upper()}  {(got==lab).sum():>3}/{len(od_list)}  {acc:5.1f}%")

print("\n[세 방식이 서로 갈리는 OD]")
split = [o for o in od_list if len({o["m1"], o["m2"], o["m3"]}) > 1]
print(f"  {len(split)}쌍")
for o in split[:10]:
    print(f"    {o['od_id']:<6} 라벨={o.get('district'):<10} "
          f"M1={o['m1']:<10} M2={o['m2']:<10} M3={o['m3']:<10} "
          f"최근접 {o['nearest_km']*1000:.0f}m")

print("\n[출발지에서 최근접 핫스팟까지 거리]")
nk = np.array([o["nearest_km"] for o in od_list])
print(f"  중앙 {np.median(nk)*1000:.0f}m  평균 {nk.mean()*1000:.0f}m  최대 {nk.max()*1000:.0f}m")
print(f"  1km 초과: {(nk>1.0).sum()}쌍 / 3km 초과: {(nk>3.0).sum()}쌍")

[OD 로드] 200쌍 (인접 100 / 분리 100)
[판정 완료] 소요 0.0초

[라벨 일치율 — 참고 지표]
  M1  161/200   80.5%
  M2  161/200   80.5%
  M3  160/200   80.0%

[세 방식이 서로 갈리는 OD]
  11쌍
    V006   라벨=gyeyang    M1=bupyeong   M2=gyeyang    M3=gyeyang    최근접 348m
    V015   라벨=gyeyang    M1=gyeyang    M2=gyeyang    M3=seogu      최근접 678m
    V027   라벨=junggu     M1=junggu     M2=michuhol   M3=junggu     최근접 441m
    V028   라벨=donggu     M1=junggu     M2=donggu     M3=junggu     최근접 1134m
    V041   라벨=bupyeong   M1=bupyeong   M2=gyeyang    M3=bupyeong   최근접 173m
    V042   라벨=gyeyang    M1=bupyeong   M2=gyeyang    M3=gyeyang    최근접 94m
    V046   라벨=donggu     M1=donggu     M2=donggu     M3=junggu     최근접 92m
    V068   라벨=bupyeong   M1=bupyeong   M2=gyeyang    M3=bupyeong   최근접 276m
    S001   라벨=donggu     M1=donggu     M2=junggu     M3=junggu     최근접 131m
    S005   라벨=michuhol   M1=yeonsu     M2=michuhol   M3=michuhol   최근접 400m

[출발지에서 최근접 핫스팟까지 거리]
  중앙 429m  평균 590m  최대 2895m
  1km 초과: 37쌍 / 3km 초과: 0쌍


In [8]:
# 셀 8: 조건별 후보 추출 + 선정 비교
#
# 비교 조건
#   labeled : 표본의 district 라벨로 필터 (v5 조건, 기준선)
#   m1/m2/m3: 좌표 판정으로 필터 (복원안)
#   nofilter: 필터 없음 (v6 조건 = 현재 운영 상태)
#
# 폴백: 군구 필터 결과가 K개 미만이면 전체 핫스팟 대상으로 되돌린다.
#       서버 algorithm.py:460-461의 동작과 대조하여 확인이 필요하다.

CONDS = ["labeled", "m1", "m2", "m3", "nofilter"]


def candidates_for(od, cond):
    """조건별 Buffer 후보와 폴백 발동 여부를 돌려준다."""
    origin, destination = od["O"], od["D"]
    if cond == "nofilter":
        return ellipse_filter(hotspots, origin, destination, 1.5), False

    d = od.get("district") if cond == "labeled" else od[cond]
    subset = hotspots[hotspots["source_district"] == d]
    r = get_buffer_radius(d)
    cands = ellipse_filter(subset, origin, destination, r) if len(subset) else subset

    if len(cands) < K:                      # 폴백
        return ellipse_filter(hotspots, origin, destination, 1.5), True
    return cands, False


rows = []
t0 = time.time()
for i, od in enumerate(od_list):
    rec = {
        "od_id": od.get("od_id"), "group": od.get("group"),
        "district_label": od.get("district"),
        "m1": od["m1"], "m2": od["m2"], "m3": od["m3"],
        "nearest_m": od["nearest_km"] * 1000,
        "direct_km": od.get("direct_km", haversine_km(od["O"], od["D"])),
    }
    sel_sets = {}
    for cond in CONDS:
        cands, fb = candidates_for(od, cond)
        rec[f"n_cand_{cond}"] = len(cands)
        rec[f"fallback_{cond}"] = fb
        sel = select_hotspots(cands, od["O"], od["D"])
        if sel is None or len(sel) < K:
            sel_sets[cond] = None
            rec[f"ok_{cond}"] = False
        else:
            sel_sets[cond] = frozenset(sel["hotspot_id"])
            rec[f"ok_{cond}"] = True
            rec[f"meanSi_{cond}"] = float(sel["S_i"].mean())

    base = sel_sets["labeled"]
    for cond in ("m1", "m2", "m3", "nofilter"):
        rec[f"same_{cond}"] = (base is not None and sel_sets[cond] == base)
        rec[f"ncommon_{cond}"] = (len(base & sel_sets[cond])
                                  if base is not None and sel_sets[cond] is not None else -1)
    rows.append(rec)
    if (i + 1) % 50 == 0:
        print(f"  진행 {i+1}/{len(od_list)}")

df = pd.DataFrame(rows)
print(f"\n[비교 완료] 소요 {time.time()-t0:.1f}초, {len(df)}행")

  진행 50/200
  진행 100/200
  진행 150/200
  진행 200/200

[비교 완료] 소요 28.4초, 200행


In [9]:
# 셀 9: 집계

print("=" * 76)
print("[주 지표] 좌표 판정 선정 결과가 라벨 기준과 같은가")
print("=" * 76)
base_ok = df["ok_labeled"]
print(f"  기준선(labeled) 선정 성공 {int(base_ok.sum())}/{len(df)}쌍")
print(f"\n  {'조건':<10} {'선정 동일':>10} {'일치율':>8} {'2곳 중 1곳 공통':>16}")
print("-" * 50)
for cond in ("m1", "m2", "m3", "nofilter"):
    sub = df[base_ok]
    same = sub[f"same_{cond}"].sum()
    one = int((sub[f"ncommon_{cond}"] == 1).sum())
    print(f"  {cond:<10} {same:>7}/{len(sub):<3} {same/len(sub)*100:>7.1f}% {one:>14}")

print()
print("=" * 76)
print("[부 지표] 후보 수 · 폴백 · K=2 충족")
print("=" * 76)
print(f"  {'조건':<10} {'후보수 평균':>11} {'중앙':>7} {'폴백':>7} {'선정 실패':>9}")
print("-" * 50)
for cond in CONDS:
    print(f"  {cond:<10} {df[f'n_cand_{cond}'].mean():>10.1f} "
          f"{df[f'n_cand_{cond}'].median():>7.0f} "
          f"{int(df[f'fallback_{cond}'].sum()):>7} "
          f"{int((~df[f'ok_{cond}']).sum()):>9}")

print("\n  참고: v5(군구 필터 적용) 평균 후보 20.4곳 / v6(필터 제거) 41.4곳")

print()
print("=" * 76)
print("[오염도] 선정 핫스팟 평균 S_i")
print("=" * 76)
for cond in CONDS:
    col = f"meanSi_{cond}"
    if col in df.columns:
        print(f"  {cond:<10} {df[col].mean():.2f}")

print()
print("=" * 76)
print("[강화군 특례 확인]")
print("=" * 76)
gh = df[df["m2"] == "ganghwa"]
print(f"  M2가 강화군으로 판정한 OD: {len(gh)}쌍")
if len(gh):
    print(f"    후보수 평균 {gh['n_cand_m2'].mean():.1f}곳 "
          f"(Buffer 반경 2.5km 적용, 폴백 {int(gh['fallback_m2'].sum())}건)")
else:
    print("    표본에 강화군 OD가 없다. 특례 동작은 셀 6의 중심 좌표 확인으로 갈음한다.")

[주 지표] 좌표 판정 선정 결과가 라벨 기준과 같은가
  기준선(labeled) 선정 성공 200/200쌍

  조건              선정 동일      일치율       2곳 중 1곳 공통
--------------------------------------------------
  m1             169/200    84.5%              0
  m2             169/200    84.5%              0
  m3             168/200    84.0%              0
  nofilter       145/200    72.5%              9

[부 지표] 후보 수 · 폴백 · K=2 충족
  조건              후보수 평균      중앙      폴백     선정 실패
--------------------------------------------------
  labeled          27.7      17       8         0
  m1               29.5      17       2         0
  m2               30.9      18       2         0
  m3               29.8      18       2         0
  nofilter         46.5      29       0         0

  참고: v5(군구 필터 적용) 평균 후보 20.4곳 / v6(필터 제거) 41.4곳

[오염도] 선정 핫스팟 평균 S_i
  labeled    2.97
  m1         3.16
  m2         3.28
  m3         3.19
  nofilter   3.59

[강화군 특례 확인]
  M2가 강화군으로 판정한 OD: 22쌍
    후보수 평균 10.5곳 (Buffer 반경 2.5km 적용, 폴백 0건)


In [10]:
# 셀 10: 판정 + 인천 경계 밖 입력 대응 임계값 + 저장

sub = df[df["ok_labeled"]]
rates = {c: sub[f"same_{c}"].mean() * 100 for c in ("m1", "m2", "m3")}
best = max(rates, key=rates.get)


def verdict(v):
    return "채택" if v >= 95 else ("조정 후 재검토" if v >= 85 else "방법 A 전환")


print("=" * 76)
print("[판정 — 셀 0의 사전 기준 적용]")
print("=" * 76)
for c in ("m1", "m2", "m3"):
    print(f"  {c.upper()} 선정 일치율 {rates[c]:5.1f}%  → {verdict(rates[c])}")
print(f"\n  최선: {best.upper()} ({rates[best]:.1f}%)")
print(f"  참고: 필터 없음(현재 운영 상태)은 라벨 기준과 "
      f"{sub['same_nofilter'].mean()*100:.1f}% 일치")

print()
print("=" * 76)
print("[인천 경계 밖 입력 대응 — 최근접 핫스팟 거리 기준]")
print("=" * 76)
nm = df["nearest_m"]
print(f"  인천 내부 표본 200쌍 기준 최근접 거리")
print(f"    중앙 {nm.median():.0f}m  95분위 {nm.quantile(0.95):.0f}m  최대 {nm.max():.0f}m")
print(f"  제안: 최근접 핫스팟이 {max(3000, int(nm.max()*1.5//500*500))}m를 넘으면")
print(f"        인천 외 지역으로 보고 요청을 거절한다 (미결 E그룹 F-07~F-11).")

out = V8_ROOT / f"district_judge_compare_{STAMP}.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
print(f"\n[저장] {out.resolve()}")

[판정 — 셀 0의 사전 기준 적용]
  M1 선정 일치율  84.5%  → 방법 A 전환
  M2 선정 일치율  84.5%  → 방법 A 전환
  M3 선정 일치율  84.0%  → 방법 A 전환

  최선: M1 (84.5%)
  참고: 필터 없음(현재 운영 상태)은 라벨 기준과 72.5% 일치

[인천 경계 밖 입력 대응 — 최근접 핫스팟 거리 기준]
  인천 내부 표본 200쌍 기준 최근접 거리
    중앙 429m  95분위 1727m  최대 2895m
  제안: 최근접 핫스팟이 4000m를 넘으면
        인천 외 지역으로 보고 요청을 거절한다 (미결 E그룹 F-07~F-11).

[저장] C:\Users\User\Documents\GitHub\ploggo-algorithm-test\output8_260920\district_judge_compare_260920.csv


In [11]:
# 셀 11 (선택): 불일치 OD 상세
# 좌표 판정이 라벨과 다른 선정을 낸 OD를 들여다본다.

cond = "m2"     # 셀 10에서 최선으로 나온 방식으로 바꿔 실행한다

bad = df[df["ok_labeled"] & ~df[f"same_{cond}"]]
print(f"[{cond.upper()} 불일치 OD] {len(bad)}쌍")
if len(bad):
    cols = ["od_id", "group", "district_label", cond, "nearest_m",
            "n_cand_labeled", f"n_cand_{cond}", f"ncommon_{cond}",
            "meanSi_labeled", f"meanSi_{cond}"]
    print(bad[cols].to_string(index=False))

    print(f"\n  군구 판정이 라벨과 달랐던 경우: "
          f"{int((bad['district_label'] != bad[cond]).sum())}/{len(bad)}쌍")
    print(f"  판정은 같은데 선정이 달랐던 경우: "
          f"{int((bad['district_label'] == bad[cond]).sum())}쌍 "
          f"(같은 군구라면 선정도 같아야 하므로 0이어야 한다)")
    print(f"  평균 S_i 변화: {bad['meanSi_labeled'].mean():.2f} → "
          f"{bad[f'meanSi_{cond}'].mean():.2f}")

    bad.to_csv(V8_ROOT / f"mismatch_{cond}_{STAMP}.csv", index=False, encoding="utf-8-sig")
    print(f"\n[저장] {(V8_ROOT / f'mismatch_{cond}_{STAMP}.csv').resolve()}")

[M2 불일치 OD] 31쌍
od_id     group district_label       m2  nearest_m  n_cand_labeled  n_cand_m2  ncommon_m2  meanSi_labeled  meanSi_m2
 V002  adjacent       michuhol   yeonsu 923.740909              43          2           0            3.25       3.50
 V010  adjacent         donggu   junggu  97.410507              23         24           0            2.75       3.00
 V014  adjacent       bupyeong  gyeyang 136.176293              17         62           0            1.00       6.00
 V020  adjacent       michuhol    seogu 119.094246              60         80           0            3.50       7.00
 V023  adjacent       bupyeong  gyeyang  34.280666              17         76           0            1.00       6.75
 V027  adjacent         junggu michuhol 440.836894               4          5           0            1.00       2.75
 V031  adjacent        namdong bupyeong 316.713889              28          6           0            2.50       1.25
 V032  adjacent       bupyeong michuhol 197.8168